# Feedback loop autoencoder on a folder of images

Uses `stabilityai/sd-vae-ft-mse` (full VAE) to repeatedly re-encode/decode each image in a folder, optionally injecting gaussian noise between iterations.

Set `INPUT_DIR` below to point at your folder of pictures. For every image, running the notebook saves, under `output/<image_name>/`:
- every iteration's frame as a PNG under `frames/`
- an animated `<image_name>_feedback_loop.gif` and `.mp4` of the whole run


In [ ]:
import os
from pathlib import Path
import cv2, torch, numpy as np
import imageio.v2 as imageio
from diffusers import AutoencoderKL

torch.set_num_threads(os.cpu_count())  # use all CPU cores (no GPU available on this machine)

INPUT_DIR = Path("sources")   # <-- folder of pictures to process
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

RES = 256
NUM_ITERS = 50
noise_std = 0   # gaussian noise stddev injected between iterations, set to 0 to disable
FPS = 10           # frame rate for the exported gif/mp4

vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
vae.eval()
SCALE = vae.config.scaling_factor

In [2]:
@torch.inference_mode()
def roundtrip(x):                      # x: tensor [1,3,H,W] in [0,1]
    lat = vae.encode(x * 2 - 1).latent_dist.sample() * SCALE
    return (vae.decode(lat / SCALE).sample.clamp(-1, 1) + 1) / 2

def add_noise(x, std):
    if std <= 0:
        return x
    return (x + torch.randn_like(x) * std).clamp(0, 1)

def imread_unicode(path):
    # cv2.imread fails on Windows paths with non-ASCII characters (accents, curly quotes, ...);
    # reading raw bytes ourselves and decoding sidesteps that, and works for both jpg and png.
    data = np.fromfile(str(path), dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

def imwrite_unicode(path, img):
    ext = Path(path).suffix
    ok, buf = cv2.imencode(ext, img)
    if not ok:
        raise IOError(f"Could not encode image for {path}")
    buf.tofile(str(path))

def image_to_tensor(image_bgr):
    h, w = image_bgr.shape[:2]
    side = min(h, w)
    y0, x0 = (h - side) // 2, (w - side) // 2
    square = image_bgr[y0:y0 + side, x0:x0 + side]  # center-crop to square first, so resize doesn't stretch/distort
    square = cv2.resize(square, (RES, RES), interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(square, cv2.COLOR_BGR2RGB)
    t = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0
    return t.unsqueeze(0)

def tensor_to_bgr(x):
    img = (x.squeeze(0).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

def find_images(folder):
    return sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in IMAGE_EXTS)

def process_image(image_path):
    stem = image_path.stem
    image_out_dir = OUTPUT_DIR / stem
    frames_dir = image_out_dir / "frames"
    frames_dir.mkdir(parents=True, exist_ok=True)

    image_bgr = imread_unicode(image_path)
    if image_bgr is None:
        print(f"  skipping {image_path.name}: could not read image")
        return

    state = image_to_tensor(image_bgr)
    frames_bgr = [tensor_to_bgr(state)]
    imwrite_unicode(frames_dir / "frame_000.png", frames_bgr[0])

    for i in range(1, NUM_ITERS + 1):
        state = add_noise(state, noise_std)   # inject noise before feeding back in
        state = roundtrip(state)
        frame = tensor_to_bgr(state)
        frames_bgr.append(frame)
        imwrite_unicode(frames_dir / f"frame_{i:03d}.png", frame)

    rgb_frames = [cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames_bgr]
    gif_path = image_out_dir / f"{stem}_feedback_loop.gif"
    mp4_path = image_out_dir / f"{stem}_feedback_loop.mp4"

    imageio.mimsave(gif_path, rgb_frames, fps=FPS)
    with imageio.get_writer(mp4_path, fps=FPS, codec="libx264", quality=8) as writer:
        for f in rgb_frames:
            writer.append_data(f)

    print(f"  saved {len(frames_bgr)} frames + gif/mp4 to {image_out_dir.resolve()}")

In [3]:
image_paths = find_images(INPUT_DIR)
print(f"Found {len(image_paths)} images in {INPUT_DIR.resolve()}")

for idx, image_path in enumerate(image_paths, 1):
    print(f"[{idx}/{len(image_paths)}] {image_path.name}")
    process_image(image_path)

print("Done.")

Found 11 images in C:\Users\Eric\OneDrive\Documents\Travail\Master\semester2\perso\feedback_loop_static_image\sources
[1/11] WhatsApp Image 2026-07-09 at 14.35.10.jpeg
  saved 51 frames + gif/mp4 to C:\Users\Eric\OneDrive\Documents\Travail\Master\semester2\perso\feedback_loop_static_image\output\WhatsApp Image 2026-07-09 at 14.35.10
[2/11] WhatsApp Image 2026-07-09 at 14.35.54.jpeg
  saved 51 frames + gif/mp4 to C:\Users\Eric\OneDrive\Documents\Travail\Master\semester2\perso\feedback_loop_static_image\output\WhatsApp Image 2026-07-09 at 14.35.54
[3/11] WhatsApp Image 2026-07-09 at 14.36.23.jpeg
  saved 51 frames + gif/mp4 to C:\Users\Eric\OneDrive\Documents\Travail\Master\semester2\perso\feedback_loop_static_image\output\WhatsApp Image 2026-07-09 at 14.36.23
[4/11] WhatsApp Image 2026-07-09 at 14.36.24.jpeg
  saved 51 frames + gif/mp4 to C:\Users\Eric\OneDrive\Documents\Travail\Master\semester2\perso\feedback_loop_static_image\output\WhatsApp Image 2026-07-09 at 14.36.24
[5/11] WhatsAp